
# Macedonian–English Code-Switching LLM Evaluation

This notebook evaluates three instruction-tuned models on the 895-example Belebele-derived dataset:

- `Qwen/Qwen2.5-7B-Instruct`
- `mistralai/Mistral-7B-Instruct-v0.3`
- `meta-llama/Llama-3.1-8B-Instruct`

Four conditions are evaluated for every example:

1. **EN** — original English
2. **EN→MK** — English matrix language with Macedonian noun switches
3. **MK** — original Macedonian
4. **MK→EN** — Macedonian matrix language with English noun switches

## Scoring

The notebook follows the standard EleutherAI Belebele multiple-choice formulation:

```text
P: <passage>
Q: <question>
A: <answer 1>
B: <answer 2>
C: <answer 3>
D: <answer 4>
Answer:
```

For each item, the model's conditional log-likelihood for the continuations `" A"`, `" B"`, `" C"`, and `" D"` is computed. The answer with the highest score is selected.

This avoids free-form answer parsing and is conceptually aligned with the zero-shot, log-likelihood-based Belebele evaluation used by the LM Evaluation Harness.

the same quantization setting for all three models.


In [ ]:
!pip -q install -U transformers accelerate bitsandbytes huggingface_hub sentencepiece pandas tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
d

In [ ]:
import os
import gc
import json
import math
import shutil
import inspect
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "A CUDA GPU is required. In Colab select Runtime > Change runtime type > GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU: {gpu_name}")
print(f"VRAM: {gpu_mem:.1f} GB")


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.6 GB



## 1. Persistent project folder and dataset

In [ ]:
from google.colab import drive, files

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/mk_en_csw_evaluation")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = PROJECT_DIR / "belebele_csw_final_895_generated.csv"
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    print("Upload belebele_csw_final_895_generated.csv")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No dataset was uploaded.")

    uploaded_name = next(iter(uploaded))
    uploaded_path = Path("/content") / uploaded_name

    if uploaded_name != "belebele_csw_final_895_generated.csv":
        print(f"Uploaded file: {uploaded_name}")
        print("It will be stored using the expected project filename.")

    shutil.copy2(uploaded_path, DATA_PATH)

print("Dataset:", DATA_PATH)
print("Results:", RESULTS_DIR)


Mounted at /content/drive
Dataset: /content/drive/MyDrive/mk_en_csw_evaluation/belebele_csw_final_895_generated.csv
Results: /content/drive/MyDrive/mk_en_csw_evaluation/results


In [ ]:
df = pd.read_csv(DATA_PATH)

REQUIRED_COLUMNS = [
    "id", "link", "question_number",
    "en_passage", "en_question", "en_A", "en_B", "en_C", "en_D", "en_correct",
    "mk_passage", "mk_question", "mk_A", "mk_B", "mk_C", "mk_D", "mk_correct",
    "en_mk_passage", "en_mk_question",
    "mk_en_passage", "mk_en_question",
    "en_mk_switch_count", "mk_en_switch_count",
    "en_mk_switch_ratio", "mk_en_switch_ratio",
]

missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
assert not missing, f"Missing required columns: {missing}"

assert len(df) == 895, f"Expected 895 rows, found {len(df)}"
assert df["id"].is_unique, "IDs are not unique."

DEV_IDS = {86, 458, 525, 614, 691}
assert DEV_IDS.isdisjoint(set(df["id"])), "A development ID is present in the evaluation set."

assert (df["en_correct"].astype(int) == df["mk_correct"].astype(int)).all(), \
    "English and Macedonian correct-answer labels do not match."

print("Rows:", len(df))
print("ID range:", int(df["id"].min()), "-", int(df["id"].max()))
print("Dataset checks: PASS")

df.head(2)


Rows: 895
ID range: 1 - 900
Dataset checks: PASS


,id,link,question_number,en_passage,en_question,en_A,en_B,en_C,en_D,en_correct,...,generator_model,protocol_version,generation_date,protocol_frozen,validation_status,semantic_fidelity,alignment_correctness,matrix_language_preservation,switch_plausibility,validation_notes
0,1,https://en.wikibooks.org/wiki/Accordion/Right_...,1,Make sure your hand is as relaxed as possible ...,"According to the passage, what would not be co...","For additional volume, increase the force with...",Keep unnecessary movement to a minimum in orde...,Be mindful of hitting the notes while maintain...,Increase the speed with which you operate the ...,1,...,GPT-5.6 Sol (ChatGPT),ChatGPT CSW Generation Protocol v1.0,2026-09-24,True,user_reviewed_ok,NaN,NaN,NaN,NaN,NaN
1,2,https://en.wikibooks.org/wiki/Accordion/Right_...,2,Make sure your hand is as relaxed as possible ...,"When playing the accordion, which of the follo...",More speed,More force,Less pressure,Less finger motion,1,...,GPT-5.6 Sol (ChatGPT),ChatGPT CSW Generation Protocol v1.0,2026-09-24,True,user_reviewed_ok,NaN,NaN,NaN,NaN,NaN



## 2. Hugging Face authentication


In [ ]:
from huggingface_hub import login
login()


## 3. Experimental configuration

### Quantization

`8bit` is the recommended starting point for a free Colab T4. It normally provides enough memory for 7–8B models while introducing less quantization than 4-bit inference.

Available settings:

- `"8bit"` — recommended for a 16 GB T4
- `"4bit"` — fallback if 8-bit runs out of memory
- `"none"` — FP16; use only with enough GPU memory

For a controlled comparison, **do not use different quantization modes for different models**.


In [ ]:
MODEL_SPECS = {
    # Primary models
    "qwen2.5-7b": "Qwen/Qwen2.5-7B-Instruct",
    "mistral-7b": "mistralai/Mistral-7B-Instruct-v0.3",
    "llama3.1-8b": "meta-llama/Llama-3.1-8B-Instruct",
}

QUANTIZATION = "8bit"   # "8bit", "4bit", or "none"

BATCH_SIZE = 8

LETTERS = ["A", "B", "C", "D"]

# The order keeps each mixed condition next to its monolingual baseline.
CONDITIONS = {
    "EN": {
        "passage": "en_passage",
        "question": "en_question",
        "answers": ["en_A", "en_B", "en_C", "en_D"],
        "correct": "en_correct",
        "switch_count": None,
        "switch_ratio": None,
    },
    "EN_MK": {
        "passage": "en_mk_passage",
        "question": "en_mk_question",
        "answers": ["en_A", "en_B", "en_C", "en_D"],
        "correct": "en_correct",
        "switch_count": "en_mk_switch_count",
        "switch_ratio": "en_mk_switch_ratio",
    },
    "MK": {
        "passage": "mk_passage",
        "question": "mk_question",
        "answers": ["mk_A", "mk_B", "mk_C", "mk_D"],
        "correct": "mk_correct",
        "switch_count": None,
        "switch_ratio": None,
    },
    "MK_EN": {
        "passage": "mk_en_passage",
        "question": "mk_en_question",
        "answers": ["mk_A", "mk_B", "mk_C", "mk_D"],
        "correct": "mk_correct",
        "switch_count": "mk_en_switch_count",
        "switch_ratio": "mk_en_switch_ratio",
    },
}

print("Models:")
for key, model_id in MODEL_SPECS.items():
    print(f"  {key}: {model_id}")

print("\nQuantization:", QUANTIZATION)
print("Batch size:", BATCH_SIZE)


Models:
  qwen2.5-7b: Qwen/Qwen2.5-7B-Instruct
  mistral-7b: mistralai/Mistral-7B-Instruct-v0.3
  llama3.1-8b: meta-llama/Llama-3.1-8B-Instruct
  qwen2.5-3b: Qwen/Qwen2.5-3B-Instruct
  llama3.2-3b: meta-llama/Llama-3.2-3B-Instruct
  allam-7b: humain-ai/ALLaM-7B-Instruct-preview

Quantization: 8bit
Batch size: 8



## 4. Prompt construction

The same prompt format is used for all models and all four language conditions.

No prompt tells the model that the text is code-switched. This avoids adding an additional mitigation cue to the main experiment.


In [ ]:
def build_prompt(row, condition):
    cfg = CONDITIONS[condition]

    passage = str(row[cfg["passage"]])
    question = str(row[cfg["question"]]).strip()
    a, b, c, d = [str(row[col]) for col in cfg["answers"]]

    return (
        f"P: {passage}\n"
        f"Q: {question}\n"
        f"A: {a}\n"
        f"B: {b}\n"
        f"C: {c}\n"
        f"D: {d}\n"
        f"Answer:"
    )


def gold_letter(row, condition):
    correct_num = int(row[CONDITIONS[condition]["correct"]])
    assert correct_num in {1, 2, 3, 4}
    return LETTERS[correct_num - 1]


# Inspect one example in all four conditions.
example = df.iloc[0]
for condition in CONDITIONS:
    print("=" * 100)
    print(condition)
    print(build_prompt(example, condition))
    print("GOLD:", gold_letter(example, condition))


EN
P: Make sure your hand is as relaxed as possible while still hitting all the notes correctly - also try not to make much extraneous motion with your fingers. This way, you will tire yourself out as little as possible. Remember there's no need to hit the keys with a lot of force for extra volume like on the piano. On the accordion, to get extra volume, you use the bellows with more pressure or speed.
Q: According to the passage, what would not be considered an accurate tip for successfully playing the accordion?
A: For additional volume, increase the force with which you hit the keys
B: Keep unnecessary movement to a minimum in order to preserve your stamina
C: Be mindful of hitting the notes while maintaining a relaxed hand
D: Increase the speed with which you operate the bellows to achieve extra volume
Answer:
GOLD: A
EN_MK
P: Make sure your раката is as relaxed as possible while still hitting all the ноти correctly - also try not to make much extraneous движења with your прстите. 


## 5. Model loading and log-likelihood scoring

The standard Belebele Harness uses a space delimiter before each answer label. Therefore the candidate continuations are:

- `" A"`
- `" B"`
- `" C"`
- `" D"`

For the selected tokenizers these should each be represented by one token. The code verifies this explicitly before evaluation.

Since all four candidates are one token long, comparing their next-token conditional probabilities is exactly the required multiple-choice ranking for these answer labels.


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)


def make_quantization_config(mode):
    if mode == "8bit":
        return BitsAndBytesConfig(load_in_8bit=True)

    if mode == "4bit":
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )

    if mode == "none":
        return None

    raise ValueError(f"Unknown quantization mode: {mode}")


def load_model_and_tokenizer(model_id):
    print(f"Loading: {model_id}")

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        use_fast=True,
    )

    if tokenizer.pad_token_id is None:
        if tokenizer.eos_token_id is None:
            raise RuntimeError("Tokenizer has neither a pad token nor an EOS token.")
        tokenizer.pad_token = tokenizer.eos_token

    # Left padding keeps the final position equal to the final real prompt token
    # for every item in a batch.
    tokenizer.padding_side = "left"

    quant_config = make_quantization_config(QUANTIZATION)

    kwargs = dict(
        device_map="auto",
        low_cpu_mem_usage=True,
    )

    if quant_config is not None:
        kwargs["quantization_config"] = quant_config
        kwargs["torch_dtype"] = torch.float16
    else:
        kwargs["torch_dtype"] = torch.float16

    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    model.eval()

    # Verify that the four Harness-style candidate continuations are one token.
    candidate_texts = [" A", " B", " C", " D"]
    tokenized = [
        tokenizer.encode(text, add_special_tokens=False)
        for text in candidate_texts
    ]

    print("Candidate tokenization:")
    for text, ids in zip(candidate_texts, tokenized):
        print(repr(text), "->", ids)

    if not all(len(ids) == 1 for ids in tokenized):
        raise RuntimeError(
            "At least one answer label is not a single token. "
            "Use the generic multi-token continuation scorer instead."
        )

    candidate_token_ids = torch.tensor(
        [ids[0] for ids in tokenized],
        dtype=torch.long,
    )

    forward_params = inspect.signature(model.forward).parameters

    if "logits_to_keep" in forward_params:
        logits_kwarg = {"logits_to_keep": 1}
    elif "num_logits_to_keep" in forward_params:
        logits_kwarg = {"num_logits_to_keep": 1}
    else:
        logits_kwarg = {}
        print(
            "WARNING: this model does not expose a last-logit-only argument. "
            "Full sequence logits will be produced, which uses more memory."
        )

    return model, tokenizer, candidate_token_ids, logits_kwarg


def get_input_device(model):
    # All selected 7–8B models should fit on one quantized GPU.
    # This obtains the actual device of the first non-meta parameter.
    for param in model.parameters():
        if param.device.type != "meta":
            return param.device
    raise RuntimeError("Could not determine the model input device.")


@torch.inference_mode()
def score_prompt_batch(
    prompts,
    model,
    tokenizer,
    candidate_token_ids,
    logits_kwarg,
):
    encoded = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=False,
        add_special_tokens=True,
    )

    device = get_input_device(model)
    encoded = {k: v.to(device) for k, v in encoded.items()}

    outputs = model(
        **encoded,
        use_cache=False,
        **logits_kwarg,
    )

    # With left padding, the final sequence position is always the final
    # non-padding token. If logits_to_keep=1 is supported, shape is [B,1,V].
    last_logits = outputs.logits[:, -1, :].float()

    ids = candidate_token_ids.to(last_logits.device)

    # Exact next-token log probabilities for A/B/C/D.
    selected_logits = last_logits.index_select(dim=-1, index=ids)
    log_normalizer = torch.logsumexp(last_logits, dim=-1, keepdim=True)
    candidate_logprobs = selected_logits - log_normalizer

    return candidate_logprobs.cpu().numpy()



## 6. Resumable evaluation

Each prediction stores:

- item ID
- condition
- gold answer
- predicted answer
- correctness
- log-probability of A/B/C/D
- top-1 margin
- switch count and ratio for code-switched conditions


In [ ]:
def atomic_save_csv(frame, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)


def evaluate_model(model_key, limit_per_condition=None, suffix=""):
    if model_key not in MODEL_SPECS:
        raise KeyError(f"Unknown model key: {model_key}")

    model_id = MODEL_SPECS[model_key]
    safe_name = model_key.replace("/", "_")
    output_path = RESULTS_DIR / f"{safe_name}_{QUANTIZATION}{suffix}_predictions.csv"

    if output_path.exists():
        results = pd.read_csv(output_path)
        print(f"Resuming from {output_path}")
        print("Existing predictions:", len(results))
    else:
        results = pd.DataFrame()

    done = set()
    if len(results):
        done = set(
            zip(
                results["id"].astype(int),
                results["condition"].astype(str),
            )
        )

    model, tokenizer, candidate_ids, logits_kwarg = load_model_and_tokenizer(model_id)

    try:
        for condition in CONDITIONS:
            rows = []

            for _, row in df.iterrows():
                key = (int(row["id"]), condition)
                if key not in done:
                    rows.append(row)

            if limit_per_condition is not None:
                rows = rows[:limit_per_condition]

            print("\n" + "=" * 100)
            print(f"{model_key} | {condition}")
            print("Pending:", len(rows))

            if not rows:
                continue

            for start in tqdm(range(0, len(rows), BATCH_SIZE)):
                batch_rows = rows[start:start + BATCH_SIZE]
                prompts = [build_prompt(row, condition) for row in batch_rows]

                scores = score_prompt_batch(
                    prompts,
                    model,
                    tokenizer,
                    candidate_ids,
                    logits_kwarg,
                )

                records = []

                for row, score_vec in zip(batch_rows, scores):
                    pred_idx = int(np.argmax(score_vec))
                    pred = LETTERS[pred_idx]
                    gold = gold_letter(row, condition)

                    sorted_scores = np.sort(score_vec)
                    margin = float(sorted_scores[-1] - sorted_scores[-2])

                    cfg = CONDITIONS[condition]
                    switch_count = (
                        0
                        if cfg["switch_count"] is None
                        else int(row[cfg["switch_count"]])
                    )
                    switch_ratio = (
                        0.0
                        if cfg["switch_ratio"] is None
                        else float(row[cfg["switch_ratio"]])
                    )

                    records.append({
                        "model_key": model_key,
                        "model_id": model_id,
                        "quantization": QUANTIZATION,
                        "id": int(row["id"]),
                        "link": row["link"],
                        "question_number": int(row["question_number"]),
                        "condition": condition,
                        "gold": gold,
                        "prediction": pred,
                        "correct": int(pred == gold),
                        "logprob_A": float(score_vec[0]),
                        "logprob_B": float(score_vec[1]),
                        "logprob_C": float(score_vec[2]),
                        "logprob_D": float(score_vec[3]),
                        "top1_margin": margin,
                        "switch_count": switch_count,
                        "switch_ratio": switch_ratio,
                    })

                batch_frame = pd.DataFrame(records)
                results = pd.concat([results, batch_frame], ignore_index=True)

                results = (
                    results
                    .drop_duplicates(subset=["id", "condition"], keep="last")
                    .sort_values(["condition", "id"])
                    .reset_index(drop=True)
                )

                atomic_save_csv(results, output_path)

            condition_acc = (
                results.loc[results["condition"] == condition, "correct"]
                .mean()
            )
            print(f"{condition} accuracy: {condition_acc:.4f}")

    finally:
        # Free GPU memory before the next model.
        del model
        del tokenizer
        gc.collect()
        torch.cuda.empty_cache()

    print("\nSaved:", output_path)
    return results



## 7. Full model evaluations


The complete experiment contains:

`895 examples × 4 conditions × 3 models = 10,740 model-item-condition evaluations`.


In [ ]:
# Model 1: Qwen2.5-7B-Instruct
qwen_results = evaluate_model("qwen2.5-7b")


Loading: Qwen/Qwen2.5-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

EN_MK accuracy: 0.8838

qwen2.5-7b | MK
Pending: 895


  0%|          | 0/112 [00:00<?, ?it/s]

MK accuracy: 0.7363

qwen2.5-7b | MK_EN
Pending: 895


  0%|          | 0/112 [00:00<?, ?it/s]

MK_EN accuracy: 0.7285

Saved: /content/drive/MyDrive/mk_en_csw_evaluation/results/qwen2.5-7b_8bit_predictions.csv


In [ ]:
# Model 2: Mistral-7B-Instruct-v0.3
mistral_results = evaluate_model("mistral-7b")


Loading: mistralai/Mistral-7B-Instruct-v0.3


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

...

EN_MK accuracy: 0.7832

mistral-7b | MK
Pending: 895


  0%|          | 0/112 [00:00<?, ?it/s]

MK accuracy: 0.5899

mistral-7b | MK_EN
Pending: 895


  0%|          | 0/112 [00:00<?, ?it/s]

MK_EN accuracy: 0.5765

Saved: /content/drive/MyDrive/mk_en_csw_evaluation/results/mistral-7b_8bit_predictions.csv


In [ ]:
# Model 3: Llama-3.1-8B
llama_results = evaluate_model("llama3.1-8b")

Loading: meta-llama/Llama-3.1-8B-Instruct


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

EN_MK accuracy: 0.8704

llama3.1-8b | MK
Pending: 895


  0%|          | 0/112 [00:00<?, ?it/s]

MK accuracy: 0.7899

llama3.1-8b | MK_EN
Pending: 895


  0%|          | 0/112 [00:00<?, ?it/s]

MK_EN accuracy: 0.7821

Saved: /content/drive/MyDrive/mk_en_csw_evaluation/results/llama3.1-8b_8bit_predictions.csv
